# LightGBM on MMELON Embeddings with K-Fold Cross-Validation

Train a LightGBM classifier on one MMELON embedding view (text, image, graph or fused) and measure its
performance with k-fold cross-validation, using the same fold indices as
[MMELON_finetuning_cv.ipynb](MMELON_finetuning_cv.ipynb) so the results are directly comparable.

**Environment:** run with the `biomed-multiview` kernel (needs `lightgbm`, listed in `pyproject.toml`).

**Inputs**
- An embedding folder from [MMELON_embedding_extraction.ipynb](MMELON_embedding_extraction.ipynb)
  (`output/<DATASET>/embeddings/<EMBEDDING_NAME>/`), extracted from the **same CSV** as below.
- The CSV with the labels. The embedding files contain `row_index`, `smiles` and `embedding` but no labels;
  labels are joined from this CSV by `row_index`.
- An `.npz` file of fold indices from [cluster_cv_folds.ipynb](cluster_cv_folds.ipynb) (or [stratified_cv.py](stratified_cv.py))
  (arrays `fold0_train`, `fold0_test`, `fold1_train`, ...). The number of folds is read from this file.

**Model:** `LGBMClassifier`, with a choice of hyperparameters (`LGBM_PARAMS_CHOICE`): `"default"` uses
LightGBM's defaults; `"suggested"` uses settings that worked well for other AIRCHECK models (more trees,
a lower learning rate, subsampling, regularization and a positive-class weight). The random seed is fixed in both.
With `BALANCE_CLASSES = True`, label 1 is weighted by (number of 0s) / (number of 1s) in each fold's training rows
(LightGBM's `scale_pos_weight`), replacing any value in the chosen hyperparameters.

**Metrics** on each fold's test rows, then mean and std over folds
- **hit@K** (hit@200, hit@100, hit@50): rank the test molecules by predicted score; hit@K is how many
  of the top K have label 1. If a test fold has fewer than K molecules, all of them are counted.
- AUROC, PR AUC, and precision, recall, specificity and accuracy at `THRESHOLD`.

**Output** (in the gitignored `aircheck/output/`):
```
output/<DATASET>/lgbm/<RUN_NAME>/
├── config.json
├── cv_metrics_per_fold.csv     # one row per fold
├── cv_summary.csv              # mean and std over folds
└── models/fold_0.txt, ...      # trained LightGBM models (load with lightgbm.Booster(model_file=...))
```

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Name of this run; outputs go to output/<DATASET>/lgbm/<RUN_NAME>/ (must not already exist)
RUN_NAME = "lgbm-image-finetuned"

# Embeddings from MMELON_embedding_extraction.ipynb, and the view to train on
EMBEDDING_DIR = Path("../output") / DATASET / "embeddings" / "train-finetuned"
VIEW = "image"

# The CSV the embeddings were extracted from (for the labels) and its fold indices
DATA_DIR = Path("../output") / DATASET / "cv_folds"   # made by cluster_cv_folds.ipynb
DATA_PATH = DATA_DIR / "train_cluster_cv_data.csv"
CV_INDICES_PATH = DATA_DIR / "train_cluster_cv_folds.npz"   # the number of folds is read from it

SMILES_COL = "SMILES"
LABEL_COL = "LABEL"

In [2]:
SEED = 42

# Balance the classes: label 1 gets weight (number of 0s) / (number of 1s), counted on each fold's own training
# rows (LightGBM's scale_pos_weight; replaces the value in LGBM_PARAMS). False = use LGBM_PARAMS as they are.
BALANCE_CLASSES = True

# LightGBM hyperparameters: "default" (LightGBM's defaults) or "suggested" (worked well for other AIRCHECK models)
LGBM_PARAMS_CHOICE = "default"
LGBM_PARAMS = {
    "default": {},
    "suggested": dict(
        scale_pos_weight=8, n_estimators=3000, learning_rate=0.02, num_leaves=31, min_child_samples=100,
        subsample=0.7, subsample_freq=1, colsample_bytree=0.6, reg_alpha=0.5, reg_lambda=5.0,
    ),
}
assert LGBM_PARAMS_CHOICE in LGBM_PARAMS, f"LGBM_PARAMS_CHOICE must be one of {list(LGBM_PARAMS)}"

THRESHOLD = 0.5           # score above this -> predicted 1 (for precision, recall, specificity, accuracy)
HIT_KS = [200, 100, 50]   # hit@K = number of label-1 molecules among the K highest-scored

In [3]:
import json


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RUN_DIR = REPO_ROOT / "aircheck" / "output" / DATASET / "lgbm" / RUN_NAME

EMBEDDING_DIR = Path(EMBEDDING_DIR).expanduser().resolve()
assert (EMBEDDING_DIR / VIEW).is_dir(), (
    f"No {VIEW!r} embeddings in {EMBEDDING_DIR}; found {[p.name for p in EMBEDDING_DIR.iterdir() if p.is_dir()]}"
)
for path in (DATA_PATH, CV_INDICES_PATH):
    assert Path(path).exists(), f"File not found: {path}"
assert not (RUN_DIR.exists() and any(RUN_DIR.iterdir())), (
    f"{RUN_DIR} already exists. Choose a new RUN_NAME or delete that folder."
)

# The embeddings must come from the same CSV, since rows are matched by position (row_index)
embedding_config = json.loads((EMBEDDING_DIR / "config.json").read_text())
if Path(embedding_config["data_path"]) != Path(DATA_PATH).resolve():
    print(f"WARNING: embeddings were extracted from {embedding_config['data_path']},\n"
          f"         not {Path(DATA_PATH).resolve()}. SMILES are checked row by row below.")

(RUN_DIR / "models").mkdir(parents=True, exist_ok=True)
config = dict(
    embedding_dir=str(EMBEDDING_DIR), view=VIEW, embedding_model=embedding_config["model"],
    data_path=str(DATA_PATH), cv_indices_path=str(CV_INDICES_PATH), smiles_col=SMILES_COL,
    label_col=LABEL_COL, seed=SEED, threshold=THRESHOLD, hit_ks=HIT_KS,
    balance_classes=BALANCE_CLASSES, lgbm_params_choice=LGBM_PARAMS_CHOICE, lgbm_params=LGBM_PARAMS[LGBM_PARAMS_CHOICE],
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Embeddings : {EMBEDDING_DIR / VIEW} (model: {embedding_config['model']})")
print(f"Labels     : {DATA_PATH}")
print(f"Folds      : {CV_INDICES_PATH}")
print(f"Output     : {RUN_DIR}")

Embeddings : ~/repos/biomed-multi-view/aircheck/output/sample-data/embeddings/train-finetuned/image (model: ~/repos/biomed-multi-view/aircheck/output/sample-data/finetuned/mmelon-final_20261020_162634/best_model.ckpt)
Labels     : ../output/sample-data/cv_folds/train_cluster_cv_data.csv
Folds      : ../output/sample-data/cv_folds/train_cluster_cv_folds.npz
Output     : ~/repos/biomed-multi-view/aircheck/output/sample-data/lgbm/lgbm-image-finetuned


## 2. Load embeddings, labels and folds

In [4]:
import numpy as np
import pandas as pd

import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)


df = read_table(DATA_PATH, [SMILES_COL, LABEL_COL])

emb = pd.read_parquet(EMBEDDING_DIR / VIEW).sort_values("row_index").set_index("row_index")
assert emb.index.is_unique and emb.index.max() < len(df), "Embedding row_index does not fit the CSV"

# Same molecule on every row? (catches embeddings extracted from a different or reordered CSV)
csv_smiles = df.loc[emb.index, SMILES_COL].astype(str).str.strip()
mismatch = (csv_smiles.values != emb["smiles"].values).sum()
assert mismatch == 0, f"{mismatch} rows have different SMILES in the embeddings and {DATA_PATH.name}"

X_all = np.stack(emb["embedding"].to_numpy()).astype(np.float32)   # rows in emb.index order
position = pd.Series(np.arange(len(emb)), index=emb.index)        # CSV row -> row of X_all
labels = df[LABEL_COL]
has_label = labels.notna()
assert set(labels[has_label].astype(int).unique()) <= {0, 1}, "labels must be 0/1"

cv_indices = np.load(CV_INDICES_PATH)
N_FOLDS = len([k for k in cv_indices.files if k.endswith("_train")])
usable = set(emb.index) & set(df.index[has_label])  # rows with an embedding and a label


def usable_rows(idx, what):
    kept = np.array([i for i in idx if i in usable], dtype=int)
    if len(kept) < len(idx):
        print(f"  {what}: skipped {len(idx) - len(kept)} rows without an embedding (invalid SMILES) or label")
    return kept


folds = []
for i in range(N_FOLDS):
    train_idx, test_idx = cv_indices[f"fold{i}_train"], cv_indices[f"fold{i}_test"]
    assert not np.intersect1d(train_idx, test_idx).size, f"fold {i}: train and test rows overlap"
    folds.append((usable_rows(train_idx, f"fold {i} train"), usable_rows(test_idx, f"fold {i} test")))

# LightGBM settings per fold: the chosen hyperparameters, plus the class weight from that fold's training rows
FOLD_PARAMS = []
for tr, _ in folds:
    params = dict(LGBM_PARAMS[LGBM_PARAMS_CHOICE])
    if BALANCE_CLASSES:
        n_pos = int(labels[tr].sum())
        n_neg = len(tr) - n_pos
        assert n_neg > 0 and n_pos > 0, "training rows need both labels to balance the classes"
        params["scale_pos_weight"] = round(n_neg / n_pos, 4)
    FOLD_PARAMS.append(params)
config["scale_pos_weight_per_fold"] = [p.get("scale_pos_weight") for p in FOLD_PARAMS]
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"{len(df):,} rows, {len(emb):,} with {VIEW} embeddings ({X_all.shape[1]} dims), {N_FOLDS} folds")
pd.DataFrame([
    dict(fold=i, train=len(tr), test=len(te),
         train_pos=int(labels[tr].sum()), test_pos=int(labels[te].sum()),
         scale_pos_weight=FOLD_PARAMS[i].get("scale_pos_weight"))
    for i, (tr, te) in enumerate(folds)
])

10,221 rows, 10,221 with image embeddings (512 dims), 1 folds


,fold,train,test,train_pos,test_pos,scale_pos_weight
0,0,7590,2600,90,100,83.3333


## 3. Train and evaluate on every fold

In [5]:
import time

import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score


def compute_metrics(y, scores) -> dict:
    y, scores = np.asarray(y).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(y[top[:k]].sum()) for k in HIT_KS}
    pred = scores > THRESHOLD
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    tn, fn = int((~pred & (y == 0)).sum()), int((~pred & (y == 1)).sum())
    both_classes = len(set(y)) == 2
    metrics.update(
        rocauc=roc_auc_score(y, scores) if both_classes else float("nan"),
        pr_auc=average_precision_score(y, scores) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(y),
    )
    return metrics


rows = []
for i, (train_idx, test_idx) in enumerate(folds):
    start = time.time()
    X_train, y_train = X_all[position[train_idx].to_numpy()], labels[train_idx].astype(int).to_numpy()
    X_test, y_test = X_all[position[test_idx].to_numpy()], labels[test_idx].astype(int).to_numpy()

    model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **FOLD_PARAMS[i])
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    model.booster_.save_model(RUN_DIR / "models" / f"fold_{i}.txt")

    rows.append(dict(fold=i, **compute_metrics(y_test, scores)))
    print(f"fold {i}: trained on {len(train_idx):,}, tested on {len(test_idx):,} in {time.time() - start:.1f}s")

cv_metrics = pd.DataFrame(rows)
cv_metrics.to_csv(RUN_DIR / "cv_metrics_per_fold.csv", index=False)
cv_metrics.round(4)

fold 0: trained on 7,590, tested on 2,600 in 1.2s


,fold,hit@200,hit@100,hit@50,rocauc,pr_auc,precision,recall,specificity,accuracy
0,0,22,16,10,0.6301,0.0999,0.2727,0.03,0.9968,0.9596


## 4. Cross-validation summary

In [6]:
metric_names = [c for c in cv_metrics.columns if c != "fold"]
cv_summary = cv_metrics[metric_names].agg(["mean", "std"]).T.rename_axis("metric").reset_index()
cv_summary.to_csv(RUN_DIR / "cv_summary.csv", index=False)

print(f"LightGBM on {VIEW} embeddings, {N_FOLDS}-fold CV:")
for _, r in cv_summary.iterrows():
    print(f"  {r['metric']:12s} {r['mean']:.4f}  (std {r['std']:.4f})")

LightGBM on image embeddings, 1-fold CV:
  hit@200      22.0000  (std nan)
  hit@100      16.0000  (std nan)
  hit@50       10.0000  (std nan)
  rocauc       0.6301  (std nan)
  pr_auc       0.0999  (std nan)
  precision    0.2727  (std nan)
  recall       0.0300  (std nan)
  specificity  0.9968  (std nan)
  accuracy     0.9596  (std nan)


In [7]:
# Files produced
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(RUN_DIR)):30s} {path.stat().st_size / 1e6:8.2f} MB")

config.json                        0.00 MB
cv_metrics_per_fold.csv            0.00 MB
cv_summary.csv                     0.00 MB
models/fold_0.txt                  0.38 MB
